In [1]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
# The usual start

load_dotenv(override=True)
openai = OpenAI()

In [3]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [4]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [5]:
push("HEY!!")

Push: HEY!!


In [ ]:
# Tools

def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}


def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [14]:
# JSON scheme for tools

record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters":{
        "type": "object",
        "properties": {
            "email":
                {"type": "string", "description": "The email address of the interested user"},
            "name":
                {"type": "string", "description": "The name of the interested user"},
            "notes":
                {"type": "string", "description": "Any additional notes about the user or their interests"}
        },
        "required": ["email"],
        "additionalProperties": False
        }
    }

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Use this tool to record any question that I couldn't answer so that I can improve in the future",
    "parameters":{
        "type": "object",
        "properties": {
            "question":
                {"type": "string", "description": "The question that I couldn't answer"}    
         },
        "required": ["question"],
        "additionalProperties": False
        }
    }


tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of the interested user'},
     'name': {'type': 'string',
      'description': 'The name of the interested user'},
     'notes': {'type': 'string',
      'description': 'Any additional notes about the user or their interests'}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Use this tool to record any question that I couldn't answer so that I can improve in the future",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that I couldn't answer"}},
    'required': ['question'],
    'additionalProperties': F

In [17]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "toold_call_id": tool_call.id})
    
    return results